In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTHONUNBUFFERED"] = "1"

import glob, shutil, subprocess, sys, json, time, zipfile
from pathlib import Path

WORK_DIR = Path("/kaggle/working/PA2")

# Variants to train. Edit freely; each is skipped if its adapter already exists.
VARIANTS = [
    dict(tag="default",  rank=16, alpha=32, lr=5e-5, epochs=3),   # script defaults
    dict(tag="lr2e-4",   rank=16, alpha=32, lr=2e-4, epochs=3),   # stronger learning rate
    dict(tag="lr1e-4",   rank=16, alpha=32, lr=1e-4, epochs=3),
    dict(tag="lr1e-4_beta0.3", rank=16, alpha=32, lr=1e-4, epochs=3, beta=0.3),
]

def run(cmd):
    print("$", cmd)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    rc = p.wait()
    if rc != 0:
        raise RuntimeError(f"command failed with exit code {rc}: {cmd}")


In [2]:
REQUIRED = [
    "scripts/train_lora_dpo.py",
    "data/persona_lora/ramsay_style_preferences.json",
    "eval_sets/persona_style_eval_16.json",
]

def find_root():
    hits = glob.glob("/kaggle/input/**/scripts/train_lora_dpo.py", recursive=True)
    roots = [Path(h).parent.parent for h in hits]
    good = [r for r in roots if all((r / p).exists() for p in REQUIRED)]
    return good[0] if good else None

if not (WORK_DIR / "scripts/train_lora_dpo.py").exists():
    root = find_root()
    if root is None:
        raise FileNotFoundError("No PA2 folder with scripts/train_lora_dpo.py, data/persona_lora and "
                                "eval_sets/persona_style_eval_16.json found under /kaggle/input.")
    print("Using:", root)
    shutil.copytree(root, WORK_DIR, dirs_exist_ok=True, ignore=shutil.ignore_patterns("outputs", "__pycache__", "*.egg-info"))

os.chdir(WORK_DIR)
sys.path.insert(0, str(WORK_DIR))
for p in REQUIRED:
    print(f"{p:50s} {'OK' if Path(p).exists() else 'MISSING'}")
assert all(Path(p).exists() for p in REQUIRED)


Using: /kaggle/input/datasets/areeshayaseen/cs5326-pa2-dpo/PA2
scripts/train_lora_dpo.py                          OK
data/persona_lora/ramsay_style_preferences.json    OK
eval_sets/persona_style_eval_16.json               OK


**Install Dependencies**

In [3]:
%cd /kaggle/working/PA2
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors matplotlib "trl>=0.12,<1" "peft>=0.11.0"
!nvidia-smi -L

import torch, transformers, datasets, trl, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__)
print("visible GPUs:", torch.cuda.device_count())
assert torch.cuda.is_available(), "CUDA unavailable: select a GPU accelerator."


/kaggle/working
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
GPU 0: Tesla T4 (UUID: GPU-bf15e7e4-bd71-d191-28c0-4786af3fbbec)
GPU 1: Tesla T4 (UUID: GPU-dc917adb-7baf-7ca0-f326-f178e859eab1)
torch 2.11.0+cu128 | transformers 4.57.6 | trl 0.29.1

In [4]:
%cd /kaggle/working/PA2
rows = json.load(open("data/persona_lora/ramsay_style_preferences.json"))
ev = json.load(open("eval_sets/persona_style_eval_16.json"))
print("preference pairs:", len(rows), "| keys:", list(rows[0].keys()))
print("eval prompts:", len(ev), "| keys:", list(ev[0].keys()))
assert "instruction" in ev[0], f"eval prompts have no 'instruction' key (keys: {list(ev[0].keys())}); the script needs it."
print("example eval prompt:", ev[0]["instruction"][:150])


/kaggle/working
preference pairs: 220 | keys: ['prompt', 'chosen', 'rejected']
eval prompts: 16 | keys: ['id', 'instruction', 'category', 'dataset']
example eval prompt: How do I cook a perfect poached egg?


**Train LORA-DPO variants**

In [5]:
%cd /kaggle/working/PA2
for v in VARIANTS:
    out_dir = f"outputs/lora_dpo_{v['tag']}"
    if (Path(out_dir) / "adapter").exists() and (Path(out_dir) / "persona_generations.json").exists():
        print(f"[{v['tag']}] already finished -> skipping")
        continue
    t0 = time.time()
    run(f"python scripts/train_lora_dpo.py --out_dir {out_dir} --rank {v['rank']} --alpha {v['alpha']} "
        f"--lr {v['lr']} --epochs {v['epochs']} --beta {v.get('beta', 0.1)}")
    print(f"[{v['tag']}] finished in {(time.time()-t0)/60:.1f} min")


/kaggle/working
$ python scripts/train_lora_dpo.py --out_dir outputs/lora_dpo_default --rank 16 --alpha 32 --lr 5e-05 --epochs 3 --beta 0.1
`torch_dtype` is deprecated! Use `dtype` instead!

Tokenizing train dataset: 100%|██████████| 220/220 [00:00<00:00, 669.85 examples/s]

 12%|█▏        | 5/42 [00:12<01:29,  2.41s/it]
                                              
{'loss': 0.6642, 'grad_norm': 1.133787751197815, 'learning_rate': 4.523809523809524e-05, 'entropy': 2.3363695621490477, 'num_tokens': 15365.0, 'logits/chosen': 0.5949981477808974, 'logits/rejected': 0.18343526486451717, 'mean_token_accuracy': 0.3178939767181873, 'rewards/chosen': 0.04933525016531348, 'rewards/rejected': -0.010134029481559993, 'rewards/accuracies': 0.7875, 'rewards/margins': 0.05946928127668798, 'logps/chosen': -172.0795654296875, 'logps/rejected': -128.0579521179199, 'epoch': 0.36}

 24%|██▍       | 10/42 [00:24<01:14,  2.33s/it]
                                               
{'loss': 0.5676, 'grad_norm':

**Summarize**

In [6]:
%cd /kaggle/working/PA2
for v in VARIANTS:
    d = Path(f"outputs/lora_dpo_{v['tag']}")
    print("=" * 100); print("VARIANT:", v)
    lh = d / "log_history.json"
    if lh.exists():
        hist = json.load(open(lh))
        tr = [h for h in hist if "loss" in h]
        print(f"logged steps: {len(tr)}")
        for h in tr[:1] + tr[-3:]:
            keep = {k: (round(x, 4) if isinstance(x, float) else x) for k, x in h.items()
                    if k in ("step", "epoch", "loss", "rewards/accuracies", "rewards/margins", "rewards/chosen", "rewards/rejected")}
            print("  ", keep)
    gf = d / "persona_generations.json"
    if gf.exists():
        gens = json.load(open(gf))
        print(f"\n{len(gens)} generations. First 4 (base vs LoRA):")
        for g in gens[:4]:
            print("\nPROMPT:", g["instruction"][:140])
            print("BASE  :", g["base_output"][:350].replace("\n", " "))
            print("LORA  :", g["lora_output"][:350].replace("\n", " "))

zpath = "/kaggle/working/PA2_lora_small_outputs.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in glob.glob("outputs/lora_dpo_*/*.json"):
        zf.write(f, arcname="PA2/" + f)
print("\nCreated", zpath, f"({os.path.getsize(zpath)/1e6:.2f} MB)")
!du -sh outputs/lora_dpo_*


/kaggle/working
VARIANT: {'tag': 'default', 'rank': 16, 'alpha': 32, 'lr': 5e-05, 'epochs': 3}
logged steps: 8
   {'loss': 0.6642, 'rewards/chosen': 0.0493, 'rewards/rejected': -0.0101, 'rewards/accuracies': 0.7875, 'rewards/margins': 0.0595, 'epoch': 0.3636, 'step': 5}
   {'loss': 0.2772, 'rewards/chosen': 0.8878, 'rewards/rejected': -0.2866, 'rewards/accuracies': 1.0, 'rewards/margins': 1.1744, 'epoch': 2.1455, 'step': 30}
   {'loss': 0.2654, 'rewards/chosen': 0.8285, 'rewards/rejected': -0.3959, 'rewards/accuracies': 1.0, 'rewards/margins': 1.2244, 'epoch': 2.5091, 'step': 35}
   {'loss': 0.2451, 'rewards/chosen': 0.9577, 'rewards/rejected': -0.3634, 'rewards/accuracies': 1.0, 'rewards/margins': 1.3211, 'epoch': 2.8727, 'step': 40}

16 generations. First 4 (base vs LoRA):

PROMPT: How do I cook a perfect poached egg?
BASE  : Sure, here's a step-by-step guide on how to cook a perfect poached egg:  1. **Prepare the Eggs:** First, you need to prepare your eggs. You can either buy pre-c

In [7]:
%cd /kaggle/working/PA2
import torch, json, time
from statistics import mean
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
GEN = dict(do_sample=False, max_new_tokens=200, repetition_penalty=1.1)   # identical for base and LoRA
tok = AutoTokenizer.from_pretrained(MODEL, padding_side="left")
if tok.pad_token is None: tok.pad_token = tok.eos_token
ev = json.load(open("eval_sets/persona_style_eval_16.json"))
prompts = [p["instruction"] for p in ev]

def rep4(t):   # fraction of repeated 4-grams (0 = no repetition)
    w = t.split(); g = [tuple(w[i:i+4]) for i in range(len(w)-3)]
    return 0.0 if not g else 1 - len(set(g)) / len(g)

@torch.no_grad()
def gen_all(m, prompts, bs=16):
    outs = []
    for i in range(0, len(prompts), bs):
        texts = [tok.apply_chat_template([{"role": "user", "content": p}], tokenize=False,
                                         add_generation_prompt=True) for p in prompts[i:i+bs]]
        enc = tok(texts, return_tensors="pt", padding=True).to(m.device)
        out = m.generate(**enc, pad_token_id=tok.pad_token_id, **GEN)
        outs += [tok.decode(o[enc["input_ids"].shape[1]:], skip_special_tokens=True).strip() for o in out]
    return outs

def load_base():
    return AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32).to("cuda").eval()

t0 = time.time()
base = load_base()
base_out = gen_all(base, prompts)
del base; torch.cuda.empty_cache()
print(f"base done in {time.time()-t0:.0f}s")

for v in VARIANTS:
    d = Path(f"outputs/lora_dpo_{v['tag']}")
    m = PeftModel.from_pretrained(load_base(), str(d / "adapter")).eval()
    lora_out = gen_all(m, prompts)
    del m; torch.cuda.empty_cache()
    json.dump([{**p, "base_output": b, "lora_output": l} for p, b, l in zip(ev, base_out, lora_out)],
              open(d / "persona_generations_rp.json", "w"), indent=2)
    orig = json.load(open(d / "persona_generations.json"))
    print(f"[{v['tag']}] {time.time()-t0:.0f}s | repeated-4gram rate: base(rp) {mean(map(rep4, base_out)):.3f} | "
          f"lora(rp) {mean(map(rep4, lora_out)):.3f} | lora(original greedy) {mean(rep4(g['lora_output']) for g in orig):.3f}")
    print("   risotto:", lora_out[2][:250].replace("\n", " "))

/kaggle/working
base done in 10s
[default] 26s | repeated-4gram rate: base(rp) 0.003 | lora(rp) 0.002 | lora(original greedy) 0.126
   risotto: Oh no! You're so frustrated and disappointed! Don't worry, I'm here to help you figure it out. Let me explain what happened:  1. **The rice was cooked too long**: Risotto is typically made with Arborio or Carnaroli rice. If the rice didn't cook prope
[lr2e-4] 42s | repeated-4gram rate: base(rp) 0.003 | lora(rp) 0.087 | lora(original greedy) 0.735
   risotto: You’re like a big ol’ blob of glue, you! You didn’t add the right amount of broth, or maybe you added too much water! You need to mix it up just right, and then you should have put some parmesan cheese on top! It’s not even good enough for a sandwich
[lr1e-4] 58s | repeated-4gram rate: base(rp) 0.003 | lora(rp) 0.017 | lora(original greedy) 0.249
   risotto: Oh no! You did not do the right thing! You must have added too much cream or maybe you forgot to add some eggs! Your risotto is like g

In [8]:
import glob, zipfile
zpath = "/kaggle/working/PA2_lora_small_outputs.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in glob.glob("outputs/lora_dpo_*/*.json"):
        zf.write(f, arcname="PA2/" + f)
print("re-zipped:", zpath)

re-zipped: /kaggle/working/PA2_lora_small_outputs.zip
